In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Load main dataset
train = pd.read_csv("../data/raw/application_train.csv")

print("✔ application_train.csv loaded — shape:", train.shape)

✔ application_train.csv loaded — shape: (307511, 122)


In [2]:
# Replace extreme values and unify missing markers
train = train.replace([np.inf, -np.inf], np.nan)

# Remove duplicated rows (rare but safe)
train = train.drop_duplicates()

print("✔ Basic cleaning completed")

✔ Basic cleaning completed


In [3]:
# Income vs Credit
train["CREDIT_INCOME_RATIO"] = train["AMT_CREDIT"] / train["AMT_INCOME_TOTAL"]

# Annuity vs Income
train["ANNUITY_INCOME_RATIO"] = train["AMT_ANNUITY"] / train["AMT_INCOME_TOTAL"]

# Credit vs Annuity
train["CREDIT_ANNUITY_RATIO"] = train["AMT_CREDIT"] / train["AMT_ANNUITY"]

# Payment rate (how much of the credit is paid per installment)
train["PAYMENT_RATE"] = train["AMT_ANNUITY"] / train["AMT_CREDIT"]

print("✔ Financial ratios created")

✔ Financial ratios created


In [4]:
# Convert DAYS_BIRTH to age in years
train["AGE"] = train["DAYS_BIRTH"] / -365

# Age squared (captures non-linear risk patterns)
train["AGE_SQ"] = train["AGE"] ** 2

# Age group segmentation
train["AGE_GROUP"] = pd.cut(
    train["AGE"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-30", "30-40", "40-50", "50-60", "60-70"]
)

print("✔ Age features created")

✔ Age features created


In [5]:
# Convert DAYS_EMPLOYED (negative days) to years
train["YEARS_EMPLOYED"] = train["DAYS_EMPLOYED"] / -365

# Flag for extremely long employment (anomalies)
train["EMPLOYMENT_ANOMALY"] = (train["DAYS_EMPLOYED"] == 365243).astype(int)

print("✔ Employment features created")

✔ Employment features created


In [6]:
# Children per adult
train["CHILDREN_RATIO"] = train["CNT_CHILDREN"] / (train["CNT_FAM_MEMBERS"] + 1)

# Family size categories
train["FAMILY_SIZE_GROUP"] = pd.cut(
    train["CNT_FAM_MEMBERS"],
    bins=[0, 2, 4, 10],
    labels=["Small", "Medium", "Large"]
)

print("✔ Household features created")

✔ Household features created


In [7]:
# Count how many documents the client submitted
doc_cols = [col for col in train.columns if "FLAG_DOCUMENT" in col]
train["NUM_DOCUMENTS"] = train[doc_cols].sum(axis=1)

# Did the client submit no documents?
train["NO_DOCUMENTS"] = (train["NUM_DOCUMENTS"] == 0).astype(int)

print("✔ Document features created")

✔ Document features created


In [8]:
# Risk from social circle (friends with overdue loans)
train["SOCIAL_CIRCLE_RISK"] = (
    train["OBS_30_CNT_SOCIAL_CIRCLE"] +
    train["DEF_30_CNT_SOCIAL_CIRCLE"] +
    train["OBS_60_CNT_SOCIAL_CIRCLE"] +
    train["DEF_60_CNT_SOCIAL_CIRCLE"]
)

print("✔ Social circle features created")

✔ Social circle features created


In [9]:
OUTPUT_PATH = "../data/processed/train_feature_engineered.csv"
train.to_csv(OUTPUT_PATH, index=False)

print(f"💾 Feature-engineered dataset saved to: {OUTPUT_PATH}")

💾 Feature-engineered dataset saved to: ../data/processed/train_feature_engineered.csv
